# 02. Data pipeline: PlantDoc and Citrus

What this notebook does:
1. Builds file lists for both datasets
2. **PlantDoc:** uses the provided test set, with an 80/20 train/validation split of the training folder (like the paper)
3. **Citrus:** this copy has no provided split and only 13 Melanose images, so it uses **5-fold stratified cross-validation**. Report mean ± std over the folds, not one lucky split
4. Saves all splits as CSV files in `/kaggle/working/splits` so every later run uses the same images
5. Checks for exact duplicate images across splits (leakage)
6. Builds PyTorch datasets and loaders and shows a sample batch

**Comparability note:** your Citrus test images are not the paper's, and the PlantDoc copy has 2,922 images versus the paper's 2,598. Compare your own baseline with your own improvements, not directly with the paper's numbers.

Needs: Accelerator on (for pin_memory only; the cells also run on CPU), and both datasets attached.

## 1. Config

In [ ]:
import os, random, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

SEED = 42
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
PD_ROOT = os.environ.get("PD_ROOT", "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset")
CT_ROOT = os.environ.get("CT_ROOT", "/kaggle/input/datasets/myprojectdictionary/citrus-leaf-disease-image/Citrus Leaf Disease Image")
OUT = os.environ.get("SPLIT_OUT", "/kaggle/working/splits")
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

## 2. Build and save splits

In [ ]:
def list_images(root):
    rows = []
    for c in sorted(os.listdir(root)):
        p = os.path.join(root, c)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            if f.lower().endswith(IMG_EXT):
                rows.append((os.path.join(p, f), c))
    return pd.DataFrame(rows, columns=["path", "label"])

def safe_split(df, test_size, seed):
    """Stratified split. Classes with fewer than 2 images cannot be stratified, so they stay in the first part."""
    counts = df.label.value_counts()
    rare = counts[counts < 2].index
    df_rare, df_ok = df[df.label.isin(rare)], df[~df.label.isin(rare)]
    a, b = train_test_split(df_ok, test_size=test_size, stratify=df_ok.label, random_state=seed)
    return pd.concat([a, df_rare]).reset_index(drop=True), b.reset_index(drop=True)

# ---------- PlantDoc: use the provided test set; split train 80/20 into train/val ----------
pd_train_full = list_images(f"{PD_ROOT}/train")
pd_test = list_images(f"{PD_ROOT}/test").reset_index(drop=True)
pd_train, pd_val = safe_split(pd_train_full, 0.20, SEED)
pd_classes = sorted(pd_train_full.label.unique())          # 28 classes (from train)
pd_map = {c: i for i, c in enumerate(pd_classes)}
unknown = set(pd_test.label) - set(pd_map)
assert not unknown, f"test classes missing from train: {unknown}"

pd_train.to_csv(f"{OUT}/plantdoc_train.csv", index=False)
pd_val.to_csv(f"{OUT}/plantdoc_val.csv", index=False)
pd_test.to_csv(f"{OUT}/plantdoc_test.csv", index=False)
print(f"PlantDoc: train {len(pd_train)} | val {len(pd_val)} | test {len(pd_test)} | classes {len(pd_classes)}")

# ---------- Citrus: no provided split, so 5-fold stratified CV (test = 1 fold, val = 12.5% of the rest) ----------
ct = list_images(CT_ROOT)
ct_classes = sorted(ct.label.unique())
ct_map = {c: i for i, c in enumerate(ct_classes)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ct_folds = []
for k, (tr_idx, te_idx) in enumerate(skf.split(ct, ct.label)):
    rest, test = ct.iloc[tr_idx], ct.iloc[te_idx].reset_index(drop=True)
    train, val = safe_split(rest, 0.125, SEED)
    ct_folds.append((train, val, test))
    train.to_csv(f"{OUT}/citrus_fold{k}_train.csv", index=False)
    val.to_csv(f"{OUT}/citrus_fold{k}_val.csv", index=False)
    test.to_csv(f"{OUT}/citrus_fold{k}_test.csv", index=False)
    assert not (set(train.path) & set(test.path)) and not (set(val.path) & set(test.path)) and not (set(train.path) & set(val.path))
print(f"Citrus: {len(ct)} images, classes {ct_classes}")
for k, (tr, va, te) in enumerate(ct_folds):
    print(f"  fold {k}: train {len(tr)} | val {len(va)} | test {len(te)}  | melanose in test: {(te.label == 'Melanose').sum()}")

## 3. Leakage check (exact duplicates across splits)

In [ ]:
def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dup_report(name, groups):
    """groups: dict split_name -> DataFrame. Reports identical files that appear in more than one split."""
    seen, clashes = {}, 0
    for sname, df in groups.items():
        for p in df.path:
            h = md5(p)
            if h in seen and seen[h] != sname:
                clashes += 1
            seen.setdefault(h, sname)
    print(f"{name}: {clashes} files are exact duplicates of a file in another split")

dup_report("PlantDoc train/val/test", {"train": pd_train, "val": pd_val, "test": pd_test})
tr0, va0, te0 = ct_folds[0]
dup_report("Citrus fold 0 train/val/test", {"train": tr0, "val": va0, "test": te0})

## 4. Class counts per split

In [ ]:
print("PlantDoc class counts (train / val / test):")
tab = pd.DataFrame({"train": pd_train.label.value_counts(), "val": pd_val.label.value_counts(),
                    "test": pd_test.label.value_counts()}).fillna(0).astype(int)
print(tab.to_string())
print()
print("Citrus class counts (all / fold-0 test):")
print(pd.DataFrame({"all": ct.label.value_counts(), "fold0_test": ct_folds[0][2].label.value_counts()}).fillna(0).astype(int).to_string())

## 5. PyTorch datasets and loaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, df, cmap, tf):
        self.paths = df.path.tolist()
        self.y = [cmap[l] for l in df.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.y[i]

def make_loaders(train_df, val_df, test_df, cmap, bs=32, workers=2, balanced=False):
    tr = LeafDS(train_df, cmap, train_tf)
    sampler = None
    if balanced:   # class-balanced sampling (the paper reports this setting in Table 11)
        cnt = np.bincount(tr.y, minlength=len(cmap)).astype(float)
        w = [1.0 / max(cnt[y], 1) for y in tr.y]
        sampler = WeightedRandomSampler(w, num_samples=len(tr), replacement=True)
    kw = dict(num_workers=workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, bs, shuffle=sampler is None, sampler=sampler, drop_last=True, **kw),
            DataLoader(LeafDS(val_df, cmap, eval_tf), bs, shuffle=False, **kw),
            DataLoader(LeafDS(test_df, cmap, eval_tf), bs, shuffle=False, **kw))

pd_loaders = make_loaders(pd_train, pd_val, pd_test, pd_map)
xb, yb = next(iter(pd_loaders[0]))
print("PlantDoc batch:", tuple(xb.shape), "labels:", yb[:8].tolist())

## 6. Sanity check: look at a batch

Check that images look like leaves and that titles match the disease names.

In [ ]:
import matplotlib.pyplot as plt
mean = torch.tensor(MEAN).view(3, 1, 1); std = torch.tensor(STD).view(3, 1, 1)
inv = {v: k for k, v in pd_map.items()}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, y in zip(axes.ravel(), xb[:8], yb[:8]):
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(inv[int(y)][:22], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## What to send me
- The printed split sizes and the **duplicate report** lines
- The class count tables (just the rows that look odd, if it is long)
- Whether the sample image grid looks right

Next: the baseline model and training loop. One decision is needed first (see my message).